# Thermal Systems Validation Platform -- Walkthrough

This notebook is a thin demo layer over the code in `src/`. It does not contain the model, fitting, identifiability, or MCMC logic itself -- that lives in `src/thermal_model.py`, `src/fitting.py`, `src/identifiability.py`, and `src/mcmc_pipeline.py`, so it can be tested and reused outside a notebook.


## Data Loading

Expects one CSV per run in `data_raw/`, named `<run_id>_<heater config>_<power>W[_fan].csv` (e.g. `19_H1H2H3_15W.csv`), with columns `time_s, T1_C, T2_C, T3_C, T4_C, T5_C, T6_C`.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

DATA_FOLDER = '../data_raw'
RESULTS_ROOT = '../results'


## Model Definition

The six-node LPTN and its ODE right-hand side, integrator, and closed-form steady-state solver live in `thermal_model.py`.


In [ ]:
from thermal_model import (
    PARAMETER_NAMES, SENSOR_NAMES, NODE_WEIGHTS,
    thermal_model, simulate_model, steady_state,
)

print('Parameters:', PARAMETER_NAMES)
print('Sensors:', SENSOR_NAMES)


## Parameter Fitting

Fits every CSV in `data_raw/` independently with constrained nonlinear least squares, writing one canonical `fit_results.npz` per run to `results/run<N>/`.


In [ ]:
from fitting import fit_all, fit_run

# Fit every run in data_raw/. Comment out if results/ already exists
# and you just want to re-run the analysis below.
fit_all(DATA_FOLDER, RESULTS_ROOT)


In [ ]:
# Inspect one run's fit quality
RUN_ID = 19
data = np.load(f'{RESULTS_ROOT}/run{RUN_ID}/fit_results.npz', allow_pickle=True)

print('Fitted parameters:')
for name, value in zip(data['parameter_names'], data['parameter_values']):
    print(f'  {name:8s} = {value:.4f}')
print(f"Aggregate RMSE: {float(data['rmse_total']):.4f} C")


## Identifiability Analysis

Profile likelihood: fix one parameter at a scanned value, re-optimize the rest, and see how much the fit is actually forced to get worse. A parameter whose profile barely rises over a wide range is practically non-identifiable, even when the overall RMSE looks fine.


In [ ]:
from identifiability import profile_all_parameters, get_ci

fit_path = f'{RESULTS_ROOT}/run{RUN_ID}/fit_results.npz'
profile_path = f'{RESULTS_ROOT}/run{RUN_ID}/profile_likelihood.npz'

profile_all_parameters(fit_path, profile_path)


In [ ]:
# 95% CIs for the well-identified parameters
for name in ['R_hp', 'R_cond', 'R_ext']:
    ci = get_ci(profile_path, name)
    if ci is None:
        print(f'{name}: not identifiable within the scanned range')
    else:
        lo, hi = ci
        print(f'{name}: 95% CI = [{lo:.4f}, {hi:.4f}]')


### MCMC cross-check

Samples the posterior around the same fit with `emcee`, as an independent check on the profile-likelihood conclusions: a non-identifiable parameter should show a wide/flat marginal posterior, not a tight peak. Requires `pip install emcee corner`.


In [ ]:
from mcmc_pipeline import run_mcmc

flat_samples, sampler = run_mcmc(
    fit_path,
    trace_plot_path=f'{RESULTS_ROOT}/run{RUN_ID}/mcmc_traces.png',
    corner_plot_path=f'{RESULTS_ROOT}/run{RUN_ID}/corner.png',
)


## Figures

Representative fit-quality plot: measured vs. modeled temperature for one run.


In [ ]:
measured = data['measured_temperatures']
simulated = data['simulated_temperatures']
time = data['time']
sensor_names = data['sensor_names']

fig, ax = plt.subplots(figsize=(9, 5))
for sensor, color in zip(['T2', 'T3', 'T4', 'T5'],
                          ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']):
    idx = np.where(sensor_names == sensor)[0][0]
    ax.plot(time / 60, measured[:, idx], color=color, linewidth=2, label=f'{sensor} measured')
    ax.plot(time / 60, simulated[:, idx], color=color, linestyle='--', linewidth=2, label=f'{sensor} model')

ax.set_xlabel('Time (min)'); ax.set_ylabel('Temperature (C)')
ax.set_title(f'Representative Thermal Model Fit: Run {RUN_ID}')
ax.legend(fontsize=9, ncol=2); ax.grid(True)
plt.tight_layout()
plt.savefig(f'{RESULTS_ROOT}/run{RUN_ID}/Figure1_fit_quality.png', dpi=300, bbox_inches='tight')
plt.show()
